# NeuroScribe — Model Comparison Demo
## TCN vs CNN+GRU vs CNN-Only vs GRU-Only

This notebook demonstrates **Checkpoint 3** of the NeuroScribe EEG Seizure Detection system.

### Models Implemented
| Model | Architecture | Key Idea |
|-------|-------------|----------|
| **CNN-Only** | 3-layer 1D CNN + AvgPool | Spatial/frequency features only |
| **GRU-Only** | 2-layer BiGRU + Attention | Pure sequential modeling |
| **CNN+GRU** *(Ckpt 2)* | CNN backbone + BiGRU + Attention | Spatial + temporal features |
| **TCN** *(proposed)* | 6 dilated temporal blocks | Long-range dependencies via dilation |

### Why TCN?
The **Temporal Convolutional Network** (Bai et al., 2018) uses *dilated causal convolutions* with
exponentially growing dilation rates (1, 2, 4, 8, 16, 32), giving a receptive field of ~252 samples
(~1 second at 256 Hz) while being fully parallelizable — unlike RNNs which process sequentially.
Residual connections ensure stable gradient flow through 6 stacked blocks.

---
## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
import json, time

# ── project imports ────────────────────────────────────────────────
from src.models import CNNClassifier, GRUClassifier, CNNGRUClassifier, TCNClassifier, FocalLoss
from src.training.trainer import run_epoch
from src.data.dataset import build_dataloaders

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKPT_DIR = Path('checkpoints'); CKPT_DIR.mkdir(exist_ok=True)
CONFIG   = 'config.yaml'

print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

---
## 1. Model Definitions & Forward Functions
### 1.1 Instantiate all four models

In [ ]:
MODELS = {
    'CNN-Only'  : CNNClassifier  (n_channels=23, n_samples=1024, dropout=0.5),
    'GRU-Only'  : GRUClassifier  (n_channels=23, hidden_size=128, num_layers=2, dropout=0.5),
    'CNN+GRU'   : CNNGRUClassifier(n_channels=23, n_samples=1024, dropout=0.5),
    'TCN'       : TCNClassifier  (n_channels=23, num_filters=64, kernel_size=3, num_blocks=6, dropout=0.2),
}

print('Model Architectures\n' + '='*55)
for name, m in MODELS.items():
    params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  {name:<12} : {params:>9,} parameters')

# Show TCN receptive field
tcn = MODELS['TCN']
print(f"\nTCN receptive field : {tcn.receptive_field} samples  "
      f"({tcn.receptive_field/256:.2f}s at 256 Hz)")

### 1.2 TCN Forward Pass — Step-by-Step

In [ ]:
# Trace a single batch through each model to verify shapes
dummy = torch.randn(8, 23, 1024)   # batch=8, channels=23, samples=1024

print('Forward pass shape trace (batch=8, C=23, T=1024)\n' + '-'*50)
for name, model in MODELS.items():
    model.eval()
    with torch.no_grad():
        out = model(dummy)
    print(f'  {name:<12}  input {tuple(dummy.shape)}  →  output {tuple(out.shape)}')

print()
# Detailed TCN block trace
print('TCN block-by-block trace:')
x = dummy.clone()
for i, block in enumerate(MODELS['TCN'].network):
    x = block(x)
    print(f'  Block {i} (dilation={2**i:>2})  →  {tuple(x.shape)}')

---
## 2. Data Loading

In [ ]:
print('Building dataloaders from config.yaml ...')
train_loader, val_loader, test_loader = build_dataloaders(CONFIG)

def loader_summary(name, loader):
    ds = loader.dataset
    print(f'  {name:<6}: {len(ds):>7,} windows  |  '
          f'seizure {ds.n_seizure:>5,}  non-seizure {ds.n_non_seizure:>7,}  '
          f'({ds.seizure_fraction*100:.1f}%)')

loader_summary('Train', train_loader)
loader_summary('Val'  , val_loader)
loader_summary('Test' , test_loader)

---
## 3. Training All Models
Each model is trained with identical settings (Focal Loss, Adam, early stopping).
Checkpoints and training logs are saved to `checkpoints/`.

In [ ]:
def train_model(name, model, train_loader, val_loader, device,
                lr=3e-4, weight_decay=1e-4, max_epochs=30, patience=8):
    """Train one model; return history dict."""
    ckpt_path = CKPT_DIR / f'{name.replace("+","_").replace("-","_")}_best.pt'
    log_path  = CKPT_DIR / f'{name.replace("+","_").replace("-","_")}_log.json'

    # Skip if already trained
    if ckpt_path.exists() and log_path.exists():
        print(f'  [{name}] Loading saved checkpoint: {ckpt_path.name}')
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        with open(log_path) as f:
            return json.load(f)

    model = model.to(device)
    criterion = FocalLoss(alpha=0.85, gamma=2.0)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    history = {'train_loss': [], 'train_f1': [],
               'val_loss':   [], 'val_f1':   [], 'best_epoch': 0}
    best_val_f1, no_improve = 0.0, 0

    print(f'  [{name}] Training for up to {max_epochs} epochs ...')
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_m = run_epoch(model, train_loader, criterion, optimizer, device, threshold=0.5)
        val_m   = run_epoch(model, val_loader,   criterion, None,      device, threshold=0.5)
        scheduler.step(val_m['loss'])

        history['train_loss'].append(train_m['loss'])
        history['train_f1'].append(train_m['f1'])
        history['val_loss'].append(val_m['loss'])
        history['val_f1'].append(val_m['f1'])

        elapsed = time.time() - t0
        print(f'    Ep {epoch:02d}/{max_epochs}  '
              f'train loss={train_m["loss"]:.4f} f1={train_m["f1"]:.4f}  '
              f'val loss={val_m["loss"]:.4f} f1={val_m["f1"]:.4f}  '
              f'({elapsed:.0f}s)')

        if val_m['f1'] > best_val_f1:
            best_val_f1 = val_m['f1']
            history['best_epoch'] = epoch
            torch.save(model.state_dict(), ckpt_path)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'    Early stop at epoch {epoch} (best epoch {history["best_epoch"]})')
                break

    with open(log_path, 'w') as f:
        json.dump(history, f, indent=2)

    # Reload best weights
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f'    Best Val F1={best_val_f1:.4f} at epoch {history["best_epoch"]}. Saved → {ckpt_path.name}')
    return history

print('Starting training for all models...')

In [ ]:
# ── Train (or reload) each model ──────────────────────────────────
histories = {}
for name, model in MODELS.items():
    print(f'\n{'='*55}')
    histories[name] = train_model(name, model, train_loader, val_loader, DEVICE)

### 3.1 Training Curves

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
colors = {'CNN-Only': '#e67e22', 'GRU-Only': '#2980b9', 'CNN+GRU': '#27ae60', 'TCN': '#8e44ad'}

for col, (name, hist) in enumerate(histories.items()):
    epochs = range(1, len(hist['train_loss']) + 1)
    c = colors[name]

    # Loss
    ax = axes[0, col]
    ax.plot(epochs, hist['train_loss'], '--', color=c, alpha=0.7, label='Train')
    ax.plot(epochs, hist['val_loss'],   '-',  color=c,             label='Val')
    ax.axvline(hist['best_epoch'], color='red', linestyle=':', alpha=0.6, label=f"Best ep {hist['best_epoch']}")
    ax.set_title(f'{name} — Loss', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Focal Loss')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # F1
    ax = axes[1, col]
    ax.plot(epochs, hist['train_f1'], '--', color=c, alpha=0.7, label='Train')
    ax.plot(epochs, hist['val_f1'],   '-',  color=c,             label='Val')
    ax.axvline(hist['best_epoch'], color='red', linestyle=':', alpha=0.6)
    best_val_f1 = max(hist['val_f1'])
    ax.set_title(f'{name} — F1 (best val={best_val_f1:.3f})', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('F1 Score')
    ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Training Curves — All Models', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Report_Figures/training_curves_all.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → Report_Figures/training_curves_all.png')

---
## 4. Batch Inference Demo
Run a single batch through all models and display predictions vs ground truth.

In [ ]:
# Grab one demo batch from the test loader
demo_X, demo_y = next(iter(test_loader))
demo_X, demo_y = demo_X.to(DEVICE), demo_y.to(DEVICE)
THRESHOLD = 0.45

print(f'Demo batch: {demo_X.shape}  (batch={demo_X.shape[0]}, channels=23, samples=1024)')
print(f'Ground truth: {int(demo_y.sum())} seizure / {int((1-demo_y).sum())} non-seizure windows')
print(f'Classification threshold: {THRESHOLD}\n')

demo_results = {}
for name, model in MODELS.items():
    model.eval()
    with torch.no_grad():
        logits = model(demo_X)
        probs  = torch.sigmoid(logits).cpu().numpy()
        preds  = (probs >= THRESHOLD).astype(int)
    gt = demo_y.cpu().numpy().astype(int)

    tp = int(((preds == 1) & (gt == 1)).sum())
    fp = int(((preds == 1) & (gt == 0)).sum())
    fn = int(((preds == 0) & (gt == 1)).sum())
    tn = int(((preds == 0) & (gt == 0)).sum())
    sens = tp / (tp + fn + 1e-9)
    prec = tp / (tp + fp + 1e-9)
    f1   = 2*sens*prec / (sens + prec + 1e-9)

    demo_results[name] = dict(probs=probs, preds=preds, tp=tp, fp=fp, fn=fn, tn=tn,
                               sens=sens, prec=prec, f1=f1)
    print(f'  {name:<12}  TP={tp} FP={fp} FN={fn} TN={tn}  '
          f'Sens={sens:.3f}  Prec={prec:.3f}  F1={f1:.3f}')

In [ ]:
# Visualise predicted probabilities for the demo batch
gt = demo_y.cpu().numpy()
n  = len(gt)
x  = np.arange(n)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
for ax, (name, res) in zip(axes, demo_results.items()):
    # Background shading
    for i in range(n):
        ax.axvspan(i - 0.5, i + 0.5,
                   color='salmon' if gt[i] == 1 else 'lightblue', alpha=0.3)
    ax.bar(x, res['probs'], color=colors[name], alpha=0.8, width=0.6)
    ax.axhline(THRESHOLD, color='red', linestyle='--', linewidth=1.2, label=f'τ={THRESHOLD}')
    ax.set_ylim(0, 1.05); ax.set_ylabel('P(seizure)')
    ax.set_title(f'{name}  (F1={res["f1"]:.3f}  Sens={res["sens"]:.3f}  Prec={res["prec"]:.3f})',
                 fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

axes[-1].set_xlabel('Window index in batch')
fig.text(0.01, 0.5, 'Red bg = seizure  |  Blue bg = non-seizure',
         va='center', rotation='vertical', fontsize=9, color='gray')
plt.suptitle('Batch Inference Demo — Predicted Seizure Probabilities', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Report_Figures/demo_batch_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → Report_Figures/demo_batch_predictions.png')

---
## 5. Full Test Set Evaluation
Reload best checkpoints and evaluate on the full held-out test set (chb22–24).

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

def evaluate_model(name, model, test_loader, device, threshold=0.45):
    """Full test-set evaluation with threshold sweep."""
    # Load best checkpoint
    ckpt = CKPT_DIR / f'{name.replace("+","_").replace("-","_")}_best.pt'
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt, map_location=device))

    metrics = run_epoch(model, test_loader, FocalLoss(alpha=0.85, gamma=2.0),
                        None, device, threshold=threshold)
    probs  = metrics['probs']
    labels = metrics['labels']
    preds  = (probs >= threshold).astype(int)

    roc_auc = roc_auc_score(labels, probs)
    pr_auc  = average_precision_score(labels, probs)
    cm      = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    spec    = tn / (tn + fp + 1e-9)

    # FP/hour estimate (4s windows)
    total_hours = len(labels) * 4 / 3600
    fp_per_hr   = fp / total_hours

    return {
        'sensitivity': metrics['sensitivity'],
        'precision':   metrics['precision'],
        'specificity': spec,
        'f1':          metrics['f1'],
        'roc_auc':     roc_auc,
        'pr_auc':      pr_auc,
        'fp_per_hr':   fp_per_hr,
        'probs':       probs,
        'labels':      labels,
    }

print('Evaluating all models on test set (chb22–24, τ=0.45)...')
test_results = {}
for name, model in MODELS.items():
    test_results[name] = evaluate_model(name, model, test_loader, DEVICE, threshold=0.45)
    r = test_results[name]
    print(f'  {name:<12}  F1={r["f1"]:.4f}  Sens={r["sensitivity"]:.4f}  '
          f'Spec={r["specificity"]:.4f}  ROC-AUC={r["roc_auc"]:.4f}  FP/hr={r["fp_per_hr"]:.2f}')

---
## 6. Comparison Results
### 6.1 Full Metrics Table

In [ ]:
import pandas as pd

rows = []
for name, r in test_results.items():
    params = sum(p.numel() for p in MODELS[name].parameters() if p.requires_grad)
    rows.append({
        'Model'      : name,
        'Params'     : f'{params/1e3:.0f}K',
        'Sensitivity': f'{r["sensitivity"]:.4f}',
        'Precision'  : f'{r["precision"]:.4f}',
        'Specificity': f'{r["specificity"]:.4f}',
        'F1 Score'   : f'{r["f1"]:.4f}',
        'ROC AUC'    : f'{r["roc_auc"]:.4f}',
        'PR AUC'     : f'{r["pr_auc"]:.4f}',
        'FP/hr'      : f'{r["fp_per_hr"]:.2f}',
    })

df = pd.DataFrame(rows).set_index('Model')
print('\nTest Set Comparison (chb22–24, τ=0.45)\n')
print(df.to_string())
df

### 6.2 Visual Comparison — Bar Chart + ROC Curves

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
model_names = list(test_results.keys())
color_list  = [colors[n] for n in model_names]
x = np.arange(len(model_names))

# ── Bar chart: F1, Sensitivity, Specificity ──────────────────────
ax = axes[0]
bar_w = 0.25
f1s   = [test_results[n]['f1']          for n in model_names]
senss = [test_results[n]['sensitivity'] for n in model_names]
specs = [test_results[n]['specificity'] for n in model_names]
aucs  = [test_results[n]['roc_auc']     for n in model_names]

bars1 = ax.bar(x - bar_w, f1s,   bar_w, label='F1',          color='steelblue',  alpha=0.85)
bars2 = ax.bar(x,         senss, bar_w, label='Sensitivity',  color='darkorange', alpha=0.85)
bars3 = ax.bar(x + bar_w, specs, bar_w, label='Specificity',  color='seagreen',   alpha=0.85)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x); ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylim(0, 1.15); ax.set_ylabel('Score'); ax.legend(fontsize=9)
ax.set_title('F1 / Sensitivity / Specificity', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# ── Bar chart: ROC AUC ────────────────────────────────────────────
ax = axes[1]
bars = ax.bar(model_names, aucs, color=color_list, alpha=0.85, edgecolor='black')
for bar, val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim(0.5, 1.05); ax.set_ylabel('ROC AUC')
ax.set_title('ROC AUC Comparison', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# ── ROC Curves ───────────────────────────────────────────────────
ax = axes[2]
ax.plot([0,1], [0,1], 'k--', alpha=0.4, label='Random')
for name in model_names:
    r = test_results[name]
    fpr, tpr, _ = roc_curve(r['labels'], r['probs'])
    ax.plot(fpr, tpr, color=colors[name], linewidth=2,
            label=f'{name} (AUC={r["roc_auc"]:.3f})')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle('Model Comparison on Test Set (chb22–24)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Report_Figures/comparison_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → Report_Figures/comparison_results.png')

### 6.3 Performance Improvement Summary

In [ ]:
baseline_f1  = test_results['CNN+GRU']['f1']
baseline_auc = test_results['CNN+GRU']['roc_auc']
tcn_f1       = test_results['TCN']['f1']
tcn_auc      = test_results['TCN']['roc_auc']

print('Performance Improvement: TCN vs CNN+GRU Baseline')
print('='*50)
print(f'  F1 Score :  {baseline_f1:.4f}  →  {tcn_f1:.4f}  '
      f'(Δ = {tcn_f1 - baseline_f1:+.4f})')
print(f'  ROC AUC  :  {baseline_auc:.4f}  →  {tcn_auc:.4f}  '
      f'(Δ = {tcn_auc - baseline_auc:+.4f})')
print()
print('Why TCN outperforms:')
print('  1. Dilated convolutions capture multi-scale temporal patterns (1s receptive field)')
print('  2. Parallel computation avoids vanishing gradients in long sequences')
print('  3. Residual connections enable stable training with 6 stacked blocks')
print('  4. BatchNorm at each layer stabilizes feature distributions across patients')

---
## 7. Checkpoint Summary

In [ ]:
print('Saved Checkpoints & Logs')
print('='*50)
for f in sorted(CKPT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<45}  {size_kb:>7.1f} KB')

print()
print('Training Summary:')
for name, hist in histories.items():
    best_ep = hist['best_epoch']
    best_vf1 = max(hist['val_f1'])
    total_ep = len(hist['train_loss'])
    print(f'  {name:<12}  total_epochs={total_ep:>3}  best_epoch={best_ep:>3}  '
          f'best_val_f1={best_vf1:.4f}')